In [1]:
import pandas as pd
import numpy as np

# Dataset

Cargando el dataset. Algunas correciones por hacer:
* La columnna folio debe ser tipo int. Además, asignar el mismo folio a los productos de un mismo ticket.
* Asignar la misma fecha a los productos de un mismo ticket.
* Asignar la misma caja a los productos de un mismo ticket.
* Asignar el mismo usuario a los productos de un mismo ticket.

In [19]:
data = pd.read_csv('..\\data\\ventas2025.csv')
# data = pd.read_csv('ventas.csv')
data.head()

,folio,caja,usuario,fecha,cantidad,producto,importe,valor,utilidad
0,45678.0,Caja 1,JENNY,01/01/2025,NaN,NaN,13.0,4.74,8.26
1,NaN,NaN,NaN,NaN,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26
2,45679.0,Caja 1,JENNY,01/01/2025,NaN,NaN,46.0,39.42,6.58
3,NaN,NaN,NaN,NaN,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16
4,NaN,NaN,NaN,NaN,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42


### Propagar valores: función `ffill`

Proceso para rellenar valores NaN según la fecha relativa:
```python
# Obtener indices de valores no nulos
indexes = np.array( [*data.loc[data.folio.notna()].index, len(data)] )
# Modificar valores no nulos
for i in range(indexes.size-1):
    folio = data.loc[indexes[i], 'folio']
    caja = data.loc[indexes[i], 'caja']
    usuario = data.loc[indexes[i], 'usuario']
    fecha = data.loc[indexes[i], 'fecha']
    
    data.loc[ indexes[i]:indexes[i+1] - 1, 'folio'] = folio
    data.loc[ indexes[i]:indexes[i+1] - 1, 'caja'] = caja
    data.loc[ indexes[i]:indexes[i+1] - 1, 'usuario'] = usuario
    data.loc[ indexes[i]:indexes[i+1] - 1, 'fecha'] = fecha
```
... (solución mejorable), feedback IA con función `ffill`:

In [22]:
data[['folio', 'caja', 'usuario', 'fecha']] = data[['folio', 'caja', 'usuario', 'fecha']].ffill()
data.head()

,folio,caja,usuario,fecha,cantidad,producto,importe,valor,utilidad
0,45678.0,Caja 1,JENNY,01/01/2025,NaN,NaN,13.0,4.74,8.26
1,45678.0,Caja 1,JENNY,01/01/2025,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26
2,45679.0,Caja 1,JENNY,01/01/2025,NaN,NaN,46.0,39.42,6.58
3,45679.0,Caja 1,JENNY,01/01/2025,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16
4,45679.0,Caja 1,JENNY,01/01/2025,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42


A continuación voy a eliminar los renglones que contengan *tickets* y la columna *documento*.

In [24]:
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)
data.head()

,folio,caja,usuario,fecha,cantidad,producto,importe,valor,utilidad
0,45678.0,Caja 1,JENNY,01/01/2025,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26
1,45679.0,Caja 1,JENNY,01/01/2025,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16
2,45679.0,Caja 1,JENNY,01/01/2025,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42
3,45680.0,Caja 1,JENNY,01/01/2025,2.0,TMGN COFFE Cafe De Olla Omnilife,28.0,17.70,10.30
4,45681.0,Caja 1,JENNY,01/01/2025,1.0,HUEVO CRIO CONO 30PZ,71.0,65.01,5.99


### Cambiar fecha

Cambiar formato de fecha a legible por Pandas y folio a tipo *int*:

In [26]:
# Convertir formato de fecha
data.fecha = pd.to_datetime(data['fecha'], format='%d/%m/%Y')
# Convertir la columna folio a tipo int
data.folio = data.folio.astype(int)
data.head()

,folio,caja,usuario,fecha,cantidad,producto,importe,valor,utilidad
0,45678,Caja 1,JENNY,2025-01-01,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26
1,45679,Caja 1,JENNY,2025-01-01,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16
2,45679,Caja 1,JENNY,2025-01-01,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42
3,45680,Caja 1,JENNY,2025-01-01,2.0,TMGN COFFE Cafe De Olla Omnilife,28.0,17.70,10.30
4,45681,Caja 1,JENNY,2025-01-01,1.0,HUEVO CRIO CONO 30PZ,71.0,65.01,5.99


Añadiendo columna para día
```python
# Crear columna
data['dia'] = np.nan
# Recuperar valores de fecha
values = data.Fecha.unique()
# Iterar sobre los valores y colocar el numero de dia correspondiente
i = 0
for j in values:
    data.loc[data.Fecha == j, 'dia'] = i
    i += 1
```
... (solución mejorable) feedback con IA función factorized:

In [28]:
data['dia'] = pd.factorize(data['fecha'])[0]
data.head()

,folio,caja,usuario,fecha,cantidad,producto,importe,valor,utilidad,dia
0,45678,Caja 1,JENNY,2025-01-01,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26,0
1,45679,Caja 1,JENNY,2025-01-01,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16,0
2,45679,Caja 1,JENNY,2025-01-01,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42,0
3,45680,Caja 1,JENNY,2025-01-01,2.0,TMGN COFFE Cafe De Olla Omnilife,28.0,17.70,10.30,0
4,45681,Caja 1,JENNY,2025-01-01,1.0,HUEVO CRIO CONO 30PZ,71.0,65.01,5.99,0


Añadiendo columna con el nombre del día de la semana:

In [30]:
from datetime import datetime
data['dia_semana'] = data['fecha'].apply(lambda fecha: fecha.strftime('%A'))
data.head()

,folio,caja,usuario,fecha,cantidad,producto,importe,valor,utilidad,dia,dia_semana
0,45678,Caja 1,JENNY,2025-01-01,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26,0,Wednesday
1,45679,Caja 1,JENNY,2025-01-01,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16,0,Wednesday
2,45679,Caja 1,JENNY,2025-01-01,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42,0,Wednesday
3,45680,Caja 1,JENNY,2025-01-01,2.0,TMGN COFFE Cafe De Olla Omnilife,28.0,17.70,10.30,0,Wednesday
4,45681,Caja 1,JENNY,2025-01-01,1.0,HUEVO CRIO CONO 30PZ,71.0,65.01,5.99,0,Wednesday


Reorganizar columnas:

In [32]:
data = data.reindex(columns=['folio', 'caja', 'usuario', 'dia', 'fecha', 'dia_semana',
                             'cantidad', 'producto','importe', 'valor', 'utilidad'])
data.head()

,folio,caja,usuario,dia,fecha,dia_semana,cantidad,producto,importe,valor,utilidad
0,45678,Caja 1,JENNY,0,2025-01-01,Wednesday,1.0,Encendedor Tokay 1pza,13.0,4.74,8.26
1,45679,Caja 1,JENNY,0,2025-01-01,Wednesday,1.0,Familiar XL 600hjas 4rollos,31.0,26.84,4.16
2,45679,Caja 1,JENNY,0,2025-01-01,Wednesday,1.0,Nescafe Dolca 22gr Sobre Grande,15.0,12.58,2.42
3,45680,Caja 1,JENNY,0,2025-01-01,Wednesday,2.0,TMGN COFFE Cafe De Olla Omnilife,28.0,17.70,10.30
4,45681,Caja 1,JENNY,0,2025-01-01,Wednesday,1.0,HUEVO CRIO CONO 30PZ,71.0,65.01,5.99


### Exportar datos

In [34]:
nombre = 'datos_ventas_2025_icsi'
data.to_csv(f'..\\data\\{nombre}.csv', index=False)

### Agrupar datos

In [325]:
data1 = pd.read_csv('d:\\cursos\\data_science\\data\\datos_ventas_2024_icsi.csv')
data2 = pd.read_csv('d:\\cursos\\data_science\\data\\datos_ventas_2025_icsi.csv')

In [343]:
nombre = 'datos_ventas_icsi'
data = pd.concat([data1, data2], ignore_index=True)
data
# data.to_csv(f'd:\\cursos\\data_science\\data\\{nombre}.csv', index=False)

,folio,caja,usuario,dia,fecha,dia_semana,producto,importe,valor,utilidad
0,1,Caja 1,admin,0,2024-02-21,Wednesday,Ariel Oxianillos De 500gm,28.0,25.93,2.07
1,1,Caja 1,admin,0,2024-02-21,Wednesday,Axion 720gr,29.0,23.97,5.03
2,2,Caja 1,admin,0,2024-02-21,Wednesday,Whiskas 1kg,53.0,43.79,9.21
3,3,Caja 1,admin,0,2024-02-21,Wednesday,Yomi Lala Fresa 190ml,11.0,6.50,4.50
4,4,Caja 1,admin,0,2024-02-21,Wednesday,Trikitrakes 85g,18.0,14.28,3.72
...,...,...,...,...,...,...,...,...,...,...
107509,72210,Caja 1,admin,198,2025-07-18,Friday,RECARGAS XTREME (MULTICEL-MINISUPER),20.0,19.00,1.00
107510,72211,Caja 1,admin,198,2025-07-18,Friday,Pepsi Jumbo 3Lt,38.0,31.58,6.42
107511,72211,Caja 1,admin,198,2025-07-18,Friday,Huevo Crio Individual 1pza,21.0,16.14,4.86
107512,72212,Caja 1,admin,198,2025-07-18,Friday,Clorets Menta 2.8gr 2Pastillas,1.5,0.77,0.73
